# Decision Tree Classifier

**Course**: CMOR 438 / INDE 577 — Data Science & Machine Learning  
**Dataset**: International Football Results (1872–present)  
**Author**: Neriah29  

---

## What is a Decision Tree?

Every algorithm we've built so far learned by **adjusting numbers** — weights nudged by gradient descent. A Decision Tree learns something completely different: it learns to ask **yes/no questions** about the data.

```
Is home_goals_rolling > 1.5?
        /              \
      YES               NO
       /                 \
Is away_win_rate < 0.4?   Predict: Draw/Away (0)
      /        \
    YES          NO
     /             \
Predict:        Predict:
Home Win (1)   Draw/Away (0)
```

This is both an algorithm and a visual artifact — you can literally read the tree and understand every decision it makes. That makes Decision Trees one of the most interpretable models in all of machine learning.

---

## How It Learns — Gini Impurity

At each node, the tree searches every feature and every possible threshold to find the split that best separates the two classes. "Best" is measured by **Gini impurity**:

$$\text{Gini} = 1 - \sum_i p_i^2$$

Where $p_i$ is the proportion of class $i$ in the node.

| Situation | Gini |
|---|---|
| All home wins (perfectly pure) | 0.0 |
| 50% home wins, 50% not (maximally impure) | 0.5 |
| 75% home wins | 0.375 |

The algorithm picks the split that reduces Gini the most — creating the purest possible child nodes.

### Feature Importance

Every time a feature is used to split a node, we record how much it reduced impurity, weighted by how many samples passed through that node. Sum these up across the whole tree and normalize — that gives **feature importance**: a ranking of which features the tree found most useful.

---

## Controlling Overfitting

An unconstrained tree will grow until every leaf is perfectly pure — memorizing every training sample. This is the worst possible overfitting.

The main tool: **max_depth**. Limit how many questions the tree can ask before it must make a prediction. Shallow trees generalize better but may underfit; deep trees fit training data perfectly but may overfit new data.

---

## Key Differences from Previous Algorithms

| | Logistic Regression | MLP | Decision Tree |
|---|---|---|---|
| Learning mechanism | Gradient descent | Backprop | Greedy splitting |
| Decision boundary | Linear | Non-linear curves | Rectangular regions |
| Needs feature scaling | Yes | Yes | **No** |
| Interpretable | Somewhat | No | **Very** |
| Overfitting risk | Low | Medium | High (if unconstrained) |


---
## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import sys
sys.path.insert(0, '../../src')
from football_ml.supervised_learning.decision_tree import DecisionTreeClassifier

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SEED = 42

---
## 1. Gini Impurity — Visualised

In [ ]:
p = np.linspace(0, 1, 300)
gini = 1 - p**2 - (1-p)**2

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(p, gini, color='#4878CF', linewidth=2.5)
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1, label='Max impurity (p=0.5)')
ax.axhline(0,   color='black', linestyle='-',  linewidth=0.8)
ax.fill_between(p, gini, alpha=0.1, color='#4878CF')
ax.set_xlabel('Proportion of class 1 in node', fontsize=12)
ax.set_ylabel('Gini Impurity', fontsize=12)
ax.set_title('Gini Impurity vs Class Proportion', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.annotate('Pure\n(Gini=0)', xy=(0, 0), xytext=(0.08, 0.15),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)
ax.annotate('Pure\n(Gini=0)', xy=(1, 0), xytext=(0.85, 0.15),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)
plt.tight_layout()
plt.show()

The tree always tries to find splits that push nodes toward the left or right edges of this curve — toward purity.

---
## 2. Load & Engineer Features

**Note**: Decision Trees do not require StandardScaler. The splitting criterion (Gini impurity) works on raw feature values — it doesn't measure distance or compute weighted sums.

In [ ]:
df = pd.read_csv('../../data/results.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df['home_win'] = (df['home_score'] > df['away_score']).astype(int)

def compute_team_rolling_stats(df, window=10):
    home_log = df[['date', 'home_team', 'home_score', 'away_score']].copy()
    home_log.columns = ['date', 'team', 'scored', 'conceded']
    away_log = df[['date', 'away_team', 'away_score', 'home_score']].copy()
    away_log.columns = ['date', 'team', 'scored', 'conceded']
    team_log = pd.concat([home_log, away_log]).sort_values('date').reset_index(drop=True)
    team_log['rolling_scored'] = (
        team_log.groupby('team')['scored']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    team_log['rolling_conceded'] = (
        team_log.groupby('team')['conceded']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    return team_log.drop_duplicates(subset=['date', 'team'], keep='last').set_index(['date', 'team'])

team_stats = compute_team_rolling_stats(df)

def get_stat(row, team_col, stat_col):
    try:
        return team_stats.loc[(row['date'], row[team_col]), stat_col]
    except KeyError:
        return np.nan

df['home_goals_rolling']    = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_scored'), axis=1)
df['home_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_conceded'), axis=1)
df['away_goals_rolling']    = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_scored'), axis=1)
df['away_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_conceded'), axis=1)

home_wins = df.groupby('home_team').apply(lambda g: (g['home_score'] > g['away_score']).mean()).rename('home_win_rate')
away_wins = df.groupby('away_team').apply(lambda g: (g['away_score'] > g['home_score']).mean()).rename('away_win_rate')
df = df.join(home_wins, on='home_team').join(away_wins, on='away_team')
df['neutral'] = df['neutral'].astype(int)

feature_cols = [
    'home_goals_rolling', 'away_goals_rolling',
    'home_conceded_rolling', 'away_conceded_rolling',
    'home_win_rate', 'away_win_rate', 'neutral'
]
df_clean = df[feature_cols + ['home_win']].dropna()
print(f'Dataset: {df_clean.shape[0]} matches | Home win rate: {df_clean["home_win"].mean():.1%}')

---
## 3. Prepare Data (no scaling needed)

In [ ]:
X = df_clean[feature_cols].values
y = df_clean['home_win'].values

# No StandardScaler — Decision Trees are scale-invariant
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

---
## 4. Effect of max_depth — Bias-Variance Tradeoff

In [ ]:
depths = list(range(1, 16))
train_scores, test_scores = [], []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d).fit(X_train, y_train)
    train_scores.append(m.score(X_train, y_train))
    test_scores.append(m.score(X_test, y_test))

best_depth = depths[np.argmax(test_scores)]
print(f'Best max_depth: {best_depth} (test accuracy: {max(test_scores):.3f})')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(depths, train_scores, label='Train accuracy', color='#4878CF', linewidth=2)
ax.plot(depths, test_scores,  label='Test accuracy',  color='#E87272', linewidth=2)
ax.axvline(best_depth, color='black', linestyle='--', linewidth=1.2, label=f'Best depth={best_depth}')
ax.set_xlabel('max_depth', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy vs Tree Depth — Bias-Variance Tradeoff', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

This plot shows the bias-variance tradeoff clearly:
- **Very shallow** (depth 1-2): both train and test accuracy are low — underfitting
- **Medium depth**: test accuracy peaks — good generalization
- **Very deep**: train accuracy → 100%, test accuracy drops — overfitting

---
## 5. Train Final Model

In [ ]:
model = DecisionTreeClassifier(
    max_depth=best_depth,
    min_samples_split=10,
    min_samples_leaf=5
).fit(X_train, y_train)

print(f'Tree depth:    {model.get_depth()}')
print(f'Leaf nodes:    {model.get_n_leaves()}')
print(f'Train accuracy: {model.score(X_train, y_train):.3f}')
print(f'Test  accuracy: {model.score(X_test,  y_test):.3f}')

---
## 6. Feature Importance

In [ ]:
importances = model.feature_importances_
sorted_idx  = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(
    [feature_cols[i] for i in sorted_idx],
    importances[sorted_idx],
    color='#4878CF', edgecolor='white'
)
ax.set_xlabel('Feature Importance (Gini reduction)', fontsize=12)
ax.set_title('Decision Tree — Feature Importances', fontsize=13, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

Feature importance tells us which features the tree relied on most to reduce impurity across all its splits. Unlike Logistic Regression weights, importances are always positive — they measure *how useful* a feature was, not which direction it pushes the prediction.

---
## 7. Confusion Matrix

In [ ]:
y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: 0', 'Pred: 1'],
            yticklabels=['True: 0', 'True: 1'], ax=ax)
ax.set_title(f'Confusion Matrix — Decision Tree (depth={best_depth})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=['Draw/Away', 'Home Win']))

---
## 8. Discussion — Does a Decision Tree Suit Football?

### What it does well
- **Interpretability**: you can literally read the tree — "if home team averaged more than 1.8 goals recently AND away team win rate is below 35%, predict home win." No other algorithm in this course gives you that.
- **No scaling required**: works directly on raw feature values
- **Feature importance**: clear ranking of which factors mattered most
- **Non-linear boundaries**: rectangular decision regions can capture complex patterns

### What it struggles with
1. **High variance**: small changes in training data can produce very different trees
2. **Axis-aligned boundaries**: splits are always perpendicular to one feature axis — diagonal patterns require many splits
3. **Overfitting**: without depth control, grows to perfectly memorize training data
4. **Football noise**: the same feature values can lead to very different outcomes in football — a tree struggles with this inherent randomness

### Why this matters for what's next

The Decision Tree's high variance (instability) is its biggest weakness. The fix? Train many trees and average them. That's **Random Forest** — and it fixes almost every weakness of a single tree while keeping the strengths. Decision Trees are the foundation; Ensemble Methods are the upgrade.

---
## Summary

| | |
|---|---|
| **Algorithm type** | Supervised, binary classification |
| **Learning mechanism** | Greedy recursive splitting (Gini impurity) |
| **Decision boundary** | Rectangular regions (axis-aligned splits) |
| **Needs feature scaling** | No |
| **Key hyperparameter** | max_depth — controls bias-variance tradeoff |
| **Key strength** | Interpretability, feature importance |
| **Key weakness** | High variance, overfitting risk |
| **Football suitability** | Moderate — interpretable but unstable |
| **Key concept introduced** | Gini impurity, feature importance, greedy splitting |
